In [1]:
import numpy as np
import os

In [5]:
# Probably won't change

username                        = "jsolisle"
path2carputils                  = "/work/e348/e348/shared/carputils/"
carp_config_file                = "/work/e348/e348/shared/software/carpentry-system-petsc/carp.conf"
archer2_config_file             = None
path2unloading                  = f"/work/e348/e348/{username}/UNLOADING_library/"
env_folder                      = f"{path2unloading}/venv_UNLOADING_library/"
python_script_path_archer2      = f"{path2unloading}/run.py"

first_sim = 0
last_sim = 3999

HCM_case = 1
scenario_number = 53

# Particular experiment

harddrive = "/scratch-nvme/e348/e348"
platform = "archer2"
folder_experiment_name_archer          = f"HCM/{HCM_case}/scenarios/{scenario_number}"
tags_setup_file_path_archer2    = f"/{harddrive}/{username}/{folder_experiment_name_archer}/json_files/tags_lvrv_fch.json"
general_setup_file_path_archer2 = f"/{harddrive}/{username}/{folder_experiment_name_archer}/json_files/{platform}_setup.json"

# local_mesh_folder = f"/media/croderog/SeagateExpansionDrive/{folder_experiment_name}"
local_hard_drive = "/media/croderog/SeagateExpansionDrive"
folder_experiment_name =  f"HCM/{HCM_case}/scenarios/{scenario_number}_more_samples"
local_mesh_folder = f"/{local_hard_drive}/{folder_experiment_name}"
datafolder        = f"{local_mesh_folder}/data"
json_paramfolder  = f"{local_mesh_folder}/json_files"
slrm_folder  = f"{local_mesh_folder}/slrm"
general_setup_file_path_local = f"{json_paramfolder}/{platform}_setup.json"

# Patient-specific EDP lv and EDP rv ranges

In [6]:
# scaling the unloading pressures 
I_mech_file = os.path.join(datafolder,"I_mechanics_patient.txt")
if not os.path.exists(I_mech_file):
	raise Exception(I_mech_file+" not found. Please provide the mechanics ranges for your patient in this file")
I_mech = np.loadtxt(I_mech_file,dtype=float)

xlabels = np.genfromtxt(f"{datafolder}/xlabels_mechanics.txt", dtype=str)

idx_edp_lv = None
idx_edp_rv = None
for i,xl in enumerate(xlabels):
	if xl=="EDP_lv":
		idx_edp_lv = i
	if xl=="EDP_rv":
		idx_edp_rv = i

X = np.loadtxt(os.path.join(datafolder,"X_mechanics_normalised.txt"),dtype=float)

edp_lv_norm = X[:,idx_edp_lv]
edp_rv_norm = X[:,idx_edp_rv]


edp_lv = edp_lv_norm*(I_mech[4,1]-I_mech[4,0])+I_mech[4,0]
edp_rv = edp_rv_norm*(I_mech[5,1]-I_mech[5,0])+I_mech[5,0]


print("-----------------------------------------------------------------------------")
print("New pressure parameter bounds : ")
print("EDP lv : "+str(np.min(edp_lv))+" , "+str(np.max(edp_lv)))
print("EDP rv : "+str(np.min(edp_rv))+" , "+str(np.max(edp_rv)))
print("-----------------------------------------------------------------------------")
X[:,idx_edp_lv] = edp_lv
X[:,idx_edp_rv] = edp_rv


np.savetxt(os.path.join(datafolder,"X.txt"),X,fmt="%g")
X_mechanics = np.loadtxt(os.path.join(datafolder,"X_mechanics_normalised.txt"),dtype=float)


X_mechanics[:,4] = edp_lv
X_mechanics[:,5] = edp_rv
np.savetxt(os.path.join(datafolder,"X_mechanics.txt"),X_mechanics,fmt="%g")


-----------------------------------------------------------------------------
New pressure parameter bounds : 
EDP lv : 8.0001203348 , 27.99992
EDP rv : 4.100019332432 , 7.7
-----------------------------------------------------------------------------


In [7]:
cmd = f"python3 ../simulation_toolbox/generate_json_parameter_files.py"
cmd += f" --datafolder {datafolder}" 
cmd += f" --fields mechanics" 
cmd += f" --paramfolder {json_paramfolder}" 
cmd += f" --defaultfile {json_paramfolder}/default.json" 

os.system(cmd)

Generating .json files from //media/croderog/SeagateExpansionDrive/HCM/1/scenarios/53_more_samples/data...
Found files for field mechanics.
generating json file...


mkdir: cannot create directory ‘//media/croderog/SeagateExpansionDrive/HCM/1/scenarios/53_more_samples/json_files’: File exists
/home/croderog/Desktop/IC_projects/simulation_toolbox/venv_simulation_toolbox/lib/python3.9/site-packages/SIMULATION_library/simulator_utils.py:78: UserWarning: You are asking to adapt beta_1 but you are not changing the ToRORd-Land or the Courtemanche-Land parameters. This flag will be ignored.
  warnings.warn("You are asking to adapt beta_1 but you are not changing the ToRORd-Land or the Courtemanche-Land parameters. This flag will be ignored.")


Done. Saved parameter files in //media/croderog/SeagateExpansionDrive/HCM/1/scenarios/53_more_samples/json_files


0

In [11]:
# os.makedirs(slrm_folder,exist_ok=True)

cmd = f"python3 ../simulation_toolbox/write_unloading_scripts.py"
cmd += f" --setup_file {general_setup_file_path_local}" 
cmd += f" --paramfolder {json_paramfolder}" 
cmd += f" --slrmfolder {slrm_folder}" 
cmd += f" --idx1 {first_sim} --idx2 {last_sim}" 
cmd += f" --HPC_tags_file {tags_setup_file_path_archer2}" 
cmd += f" --HPC_setup_file {general_setup_file_path_archer2}"
cmd += f" --HPC_path_to_unloading_library {path2unloading}" 
cmd += f" --HPC_env_folder {env_folder}"
cmd += f" --python_script_path_archer2 {python_script_path_archer2}"

os.system(cmd)


Generating unloading scripts from //media/croderog/Elements/HCM/4/scenarios/49_more_samples/json_files...
Saving slrm files to //media/croderog/Elements/HCM/4/scenarios/49_more_samples/slrm...


0